In [2]:
"""
FASE 4: Extração do desfecho frontal e dos preditores (sem rodar o OpenSim)
============================================================================
Dissertação: LCA / cutting / OpenSim / SVM linear (PPGMSB, UNEB)

Lê os resultados já gerados pela FASE 3 em PASTA_FASE3 (uma pasta por trial:
JRA_JointReaction_ReactionLoads.sto, inverse_dynamics.sto e
grf_medidas_filtradas.mot) e as coordenadas de inverse methods do Zenodo.

DESFECHO (decisão de 11/11/2026)
  Pico do momento em VALGO transmitido pela articulação do joelho direito na
  fase de aceitação de peso (contato inicial até o pico de flexão do joelho).

CONVENÇÃO DE SINAL (verificar e descrever na Metodologia)
  knee_r_on_tibia_r_in_tibia_r_mx é o momento que o fêmur (a articulação) aplica
  sobre a tíbia, em torno do eixo x da tíbia (anterior). No referencial da tíbia
  direita, y é superior e z é lateral. Pela regra da mão direita, mx > 0 desloca
  a extremidade proximal da tíbia para lateral e a distal para medial, isto é,
  tende a levar a tíbia em VARO. A articulação aplica um momento em varo quando
  resiste a uma solicitação externa em VALGO. Assim:
      mx > 0  ->  articulação resistindo a VALGO  (desfecho: Momento_Valgo_Pico_Nm)
      mx < 0  ->  articulação resistindo a VARO   (registrado como Momento_Varo_Pico_Nm)
  O padrão dominante encontrado (mx negativo, varo) é coerente com Maniar et al.
  (2018), que relatam momento em varo predominante na aceitação de peso do
  sidestep cutting. O mecanismo classicamente associado ao LCA é o valgo.

PREDITORES (os mesmos 7, no instante do pico do desfecho)
  Ângulos: flexão do joelho; flexão, adução e rotação do quadril (graus)
  Momento de adução do quadril (Nm; ID própria, validada)
  Cisalhamento médio-lateral (fz, N) e compressão (-fy, N) da reação do joelho
  O momento frontal (mx) e a reação AP (fx) não entram como preditores.

SAÍDA
  dataset_desfecho_frontal.xlsx (compatível com fase5_validacao_svm_desfecho_frontal.py)
"""

import glob
import os
import re

import numpy as np
import pandas as pd
from scipy.signal import butter, filtfilt

PASTA_FASE3 = r'C:\JG\opensim\Resultados_FASE3'
PASTA_ZENODO = (r'C:\JG\opensim\Dados_Zenodo'
                r'\Nitschke_et_al_Change_of_Direction_Marker_Simulation')
FC_FILTRO = 15.0
LIMIAR_CONTATO = 20.0
IGNORAR_INICIO_MS = 0.0     # robustez: ignora os primeiros ms após o contato inicial
BASE = 'knee_r_on_tibia_r_in_tibia_r_'


def ler_storage(caminho):
    em_graus = None
    with open(caminho, 'r', errors='ignore') as f:
        for i, linha in enumerate(f):
            if linha.lower().startswith('indegrees'):
                em_graus = 'yes' in linha.lower()
            if 'endheader' in linha.lower():
                break
    df = pd.read_csv(caminho, sep=r'\s+', skiprows=i + 1, engine='python')
    df.columns = [c.strip() for c in df.columns]
    return df, em_graus


def passa_baixa(sinal, fs, fc=FC_FILTRO, ordem=3):
    b, a = butter(ordem, fc / (fs / 2.0))
    return filtfilt(b, a, sinal)


def processar(pasta):
    m = re.search(r'P(\d+)_trial(\d+)', os.path.basename(pasta))
    part, trial = m.group(1), m.group(2)
    jra, _ = ler_storage(os.path.join(pasta, 'JRA_JointReaction_ReactionLoads.sto'))
    id_, _ = ler_storage(os.path.join(pasta, 'inverse_dynamics.sto'))
    grf, _ = ler_storage(os.path.join(pasta, 'grf_medidas_filtradas.mot'))
    arq_coord = glob.glob(os.path.join(PASTA_ZENODO, f'Participant_{part}', '**',
                                       f'trial{trial}_inverse_methods_coordinates.mot'),
                          recursive=True)[0]
    coord, em_graus = ler_storage(arq_coord)
    # Início da janela dos autores (movimento de interesse). Em ensaios cuja SO e JRA
    # começaram antes (INICIO_ANTECIPADO_S no fase3), os quadros anteriores são descartados.
    arq_mom = glob.glob(os.path.join(PASTA_ZENODO, f'Participant_{part}', '**',
                                     f'trial{trial}_inverse_methods_moments.sto'),
                        recursive=True)[0]
    t0_autores = float(ler_storage(arq_mom)[0]['time'].min())
    jra = jra[jra['time'] >= t0_autores - 1e-6].reset_index(drop=True)

    t = jra['time'].to_numpy()
    fs_c = 1.0 / np.median(np.diff(coord['time']))

    def serie_angulo(nome):
        v = passa_baixa(coord[nome].to_numpy(), fs_c)
        if em_graus is False:
            v = np.degrees(v)
        return np.interp(t, coord['time'], v)

    joelho = serie_angulo('knee_angle_r')
    apoio = np.interp(t, grf['time'], grf['ground_force_vy']) > LIMIAR_CONTATO
    idx_apoio = np.where(apoio)[0]
    if idx_apoio.size == 0:
        raise RuntimeError('sem apoio do pé direito na janela')
    i_ic = idx_apoio[0]
    i_flex = idx_apoio[np.argmin(joelho[idx_apoio])]       # flexão é negativa
    aceit = np.zeros_like(apoio)
    aceit[i_ic:i_flex + 1] = True
    aceit &= t >= t[i_ic] + IGNORAR_INICIO_MS / 1000.0
    aceit &= apoio

    mx = jra[f'{BASE}mx'].to_numpy()
    idx = np.where(aceit)[0]
    i_pico = idx[np.argmax(mx[idx])]                        # pico em valgo
    t_pico = t[i_pico]

    linha = {
        'Participante': part,
        'Movimento': trial,
        'Momento_Valgo_Pico_Nm': float(mx[i_pico]),
        'Momento_Varo_Pico_Nm': float((-mx[idx]).max()),
        'Tempo_Pico_s': float(t_pico),
        'Contato_Inicial_s': float(t[i_ic]),
        'Pico_Flexao_Joelho_s': float(t[i_flex]),
        'Pico_pct_Aceitacao': float(100 * (t_pico - t[i_ic]) / max(t[i_flex] - t[i_ic], 1e-9)),
        'Angulo_Flexao_Joelho': float(joelho[i_pico]),
        'Angulo_Flexao_Quadril': float(serie_angulo('hip_flexion_r')[i_pico]),
        'Angulo_Aducao_Quadril': float(serie_angulo('hip_adduction_r')[i_pico]),
        'Angulo_Rotacao_Quadril': float(serie_angulo('hip_rotation_r')[i_pico]),
        'Momento_Aducao_Quadril': float(np.interp(t_pico, id_['time'],
                                                  id_['hip_adduction_r_moment'])),
        'Cisalhamento_ML': float(jra[f'{BASE}fz'].iloc[i_pico]),
        'Compressao_Y': float(-jra[f'{BASE}fy'].iloc[i_pico]),
    }
    return linha


if __name__ == '__main__':
    pastas = sorted(p for p in glob.glob(os.path.join(PASTA_FASE3, 'P*_trial*'))
                    if os.path.isdir(p)
                    and re.fullmatch(r'P\d+_trial\d+', os.path.basename(p)))   # ignora cópias como *_original
    linhas, falhas = [], []
    for p in pastas:
        try:
            linhas.append(processar(p))
        except Exception as e:
            falhas.append((os.path.basename(p), str(e)))

    df = pd.DataFrame(linhas)
    sufixo = f'_ignora{IGNORAR_INICIO_MS:g}ms' if IGNORAR_INICIO_MS else ''
    saida = os.path.join(PASTA_FASE3, f'dataset_desfecho_frontal{sufixo}.xlsx')
    df.to_excel(saida, index=False)

    print(f'Ciclos extraídos: {len(df)} | falhas: {len(falhas)}')
    for f in falhas:
        print('  FALHA', f)
    v = df['Momento_Valgo_Pico_Nm']
    print(f'Momento em valgo (Nm): mínimo {v.min():.1f} | mediana {v.median():.1f} | '
          f'máximo {v.max():.1f} | ciclos com valgo positivo: {(v > 0).sum()}/{len(df)}')
    print(f'Momento em varo (Nm): mediana {df.Momento_Varo_Pico_Nm.median():.1f}')
    print(f'Instante do pico (% da aceitação de peso): mediana '
          f'{df.Pico_pct_Aceitacao.median():.0f}%')
    print(df.round(2).to_string(index=False))
    print(f'\nArquivo salvo: {saida}')

Ciclos extraídos: 30 | falhas: 0
Momento em valgo (Nm): mínimo -4.4 | mediana 7.8 | máximo 37.4 | ciclos com valgo positivo: 27/30
Momento em varo (Nm): mediana 54.5
Instante do pico (% da aceitação de peso): mediana 7%
Participante Movimento  Momento_Valgo_Pico_Nm  Momento_Varo_Pico_Nm  Tempo_Pico_s  Contato_Inicial_s  Pico_Flexao_Joelho_s  Pico_pct_Aceitacao  Angulo_Flexao_Joelho  Angulo_Flexao_Quadril  Angulo_Aducao_Quadril  Angulo_Rotacao_Quadril  Momento_Aducao_Quadril  Cisalhamento_ML  Compressao_Y
          02      0122                  10.40                 51.89          0.65               0.63                  0.84                8.11                -19.96                  17.48                 -15.51                    6.19                   16.59           -19.54        814.83
          02      0123                  15.38                 43.86          0.58               0.56                  0.74                9.68                -27.19                  21.06             